<a href="https://colab.research.google.com/github/vashgarvit014/CelebalAssignment/blob/main/Week2_GARVIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tesla Deliveries — End-to-End ML Pipeline
**Dataset:** Tesla EA Deliveries and Production Data (2015–2025)

**Pipeline covers:** Data Cleaning → EDA → Feature Engineering → Regression Modeling → Hyperparameter Tuning → Time Series Forecasting

## 1. Imports

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.stattools import adfuller
import pmdarima as pm          # pip install pmdarima
import joblib

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
print('All imports successful.')

## 2. Load Data

In [ ]:
df = pd.read_csv('tesla_deliveries_dataset_2015_2025.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning

In [ ]:
# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# Check missing values
print('\nMissing values per column:')
print(df.isnull().sum())

In [ ]:
# Fill numeric nulls with median (robust to outliers)
for col in ['Avg_Price_USD', 'Range_km', 'Production_Units']:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

# Drop rows where target is missing — cannot impute the label
df.dropna(subset=['Estimated_Deliveries'], inplace=True)

# Create datetime column for time series
df['date'] = pd.to_datetime(df[['Year', 'Month']].assign(day=1))
df = df.sort_values('date').reset_index(drop=True)

print('Clean dataset shape:', df.shape)
print('Date range:', df['date'].min(), '→', df['date'].max())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# --- Distribution of target variable ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Estimated_Deliveries'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Estimated Deliveries')
axes[0].set_xlabel('Deliveries')

sns.boxplot(x=df['Estimated_Deliveries'], ax=axes[1], color='steelblue')
axes[1].set_title('Boxplot — Estimated Deliveries (outlier check)')

plt.tight_layout()
plt.show()

print('Skewness:', round(df['Estimated_Deliveries'].skew(), 3))

In [ ]:
# --- Deliveries over time ---
plt.figure(figsize=(14, 5))
sns.lineplot(x='date', y='Estimated_Deliveries', data=df)
plt.title('Tesla Deliveries Over Time')
plt.xlabel('Date')
plt.ylabel('Estimated Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
# --- Year-over-year growth ---
yoy = df.groupby('Year')['Estimated_Deliveries'].sum()
yoy_growth = yoy.pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
yoy.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Annual Total Deliveries')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Deliveries')

yoy_growth.dropna().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Year-over-Year Growth (%)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Growth %')
axes[1].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# --- Production vs Deliveries scatter ---
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Production_Units', y='Estimated_Deliveries', hue='Year', data=df, palette='viridis')
plt.title('Production vs Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
# --- Region-wise deliveries ---
plt.figure(figsize=(12, 5))
region_avg = df.groupby('Region')['Estimated_Deliveries'].mean().sort_values(ascending=False)
sns.barplot(x=region_avg.index, y=region_avg.values, palette='Blues_d')
plt.xticks(rotation=45, ha='right')
plt.title('Average Deliveries by Region')
plt.ylabel('Avg Estimated Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap ---
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# Delivery efficiency ratio (guard against divide-by-zero)
df['delivery_ratio'] = df['Estimated_Deliveries'] / df['Production_Units'].replace(0, np.nan)

# Price per km range
df['price_per_km'] = df['Avg_Price_USD'] / df['Range_km'].replace(0, np.nan)

# Quarter (1–4)
df['quarter'] = ((df['Month'] - 1) // 3) + 1

# Cyclical month encoding — prevents model treating Dec→Jan as a large jump
df['month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# Lag feature — previous month's deliveries (strong temporal signal)
df['lag_1'] = df['Estimated_Deliveries'].shift(1)

# Rolling 3-month average (smoothed trend signal)
df['rolling_mean_3'] = df['Estimated_Deliveries'].shift(1).rolling(window=3).mean()

# Drop rows that became NaN due to lag/rolling
df.dropna(inplace=True)

print('Features after engineering:', df.shape)
df[['delivery_ratio', 'price_per_km', 'quarter', 'month_sin', 'month_cos', 'lag_1', 'rolling_mean_3']].head()

## 6. Train / Test Split  *(split BEFORE any encoding or scaling)*

In [ ]:
TARGET    = 'Estimated_Deliveries'
DROP_COLS = [TARGET, 'date']

X = df.drop(columns=DROP_COLS)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print('Train size:', X_train.shape)
print('Test size :', X_test.shape)

## 7. Build Pipelines  *(encoder and scaler fitted on train data only)*

In [ ]:
categorical_cols = ['Region', 'Model']
numerical_cols   = [c for c in X.columns if c not in categorical_cols]

# ColumnTransformer: scales numerics, one-hot encodes categoricals
# This completely replaces LabelEncoder + pd.get_dummies
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
])

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE))
])

lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

print('Pipelines trained successfully.')

## 8. Model Evaluation + Comparison Table

In [ ]:
def evaluate_model(name, pipeline, X_tr, y_tr, X_te, y_te):
    """Returns a dict of all key metrics for one model."""
    pred = pipeline.predict(X_te)
    cv_scores = cross_val_score(pipeline, X_tr, y_tr, cv=5, scoring='r2')
    return {
        'Model'         : name,
        'R² (test)'     : round(r2_score(y_te, pred), 4),
        'R² (5-fold CV)': round(cv_scores.mean(), 4),
        'MAE'           : round(mean_absolute_error(y_te, pred), 2),
        'RMSE'          : round(np.sqrt(mean_squared_error(y_te, pred)), 2),
    }

results = []
results.append(evaluate_model('Linear Regression', lr_pipeline, X_train, y_train, X_test, y_test))
results.append(evaluate_model('Random Forest',     rf_pipeline, X_train, y_train, X_test, y_test))

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# --- Residual plots for both models ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name, pipeline in zip(axes,
                               ['Linear Regression', 'Random Forest'],
                               [lr_pipeline, rf_pipeline]):
    pred      = pipeline.predict(X_test)
    residuals = y_test - pred
    ax.scatter(pred, residuals, alpha=0.5, edgecolors='k', linewidth=0.3)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_title(f'{name} — Residuals vs Predicted')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Residual')

plt.tight_layout()
plt.show()

## 9. Hyperparameter Tuning (Random Forest)

In [ ]:
param_grid = {
    'model__n_estimators'  : [100, 200],
    'model__max_depth'     : [5, 10, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf' : [1, 2],
    'model__max_features'  : ['sqrt', 'log2'],
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',  # more interpretable than R²
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
print('Best params:', grid_search.best_params_)

In [ ]:
best_pipeline = grid_search.best_estimator_

# Add tuned RF to comparison table
results.append(evaluate_model('RF (Tuned)', best_pipeline, X_train, y_train, X_test, y_test))

results_df = pd.DataFrame(results)
print('\n=== Final Model Comparison ===')
print(results_df.to_string(index=False))

## 10. Feature Importance

In [ ]:
# Extract feature names after one-hot encoding
ohe_features   = best_pipeline.named_steps['preprocessor']\
                     .named_transformers_['cat']\
                     .get_feature_names_out(categorical_cols).tolist()
feature_names  = numerical_cols + ohe_features

importances = pd.Series(
    best_pipeline.named_steps['model'].feature_importances_,
    index=feature_names
).sort_values()

plt.figure(figsize=(10, 7))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importance — Tuned Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 11. Time Series Forecasting (SARIMA)

In [ ]:
# Aggregate monthly deliveries
ts = df.groupby('date')['Estimated_Deliveries'].sum().sort_index()

# Stationarity check using Augmented Dickey-Fuller test
adf_result = adfuller(ts.dropna())
print(f'ADF Statistic : {adf_result[0]:.4f}')
print(f'p-value       : {adf_result[1]:.4f}')
print('Series is', 'stationary ✓' if adf_result[1] < 0.05 else 'non-stationary — differencing needed')

In [ ]:
# Temporal train/test split — hold out last 6 months
HOLDOUT  = 6
ts_train = ts.iloc[:-HOLDOUT]
ts_test  = ts.iloc[-HOLDOUT:]

print(f'TS train: {ts_train.index[0].date()} → {ts_train.index[-1].date()} ({len(ts_train)} months)')
print(f'TS test : {ts_test.index[0].date()}  → {ts_test.index[-1].date()}  ({len(ts_test)} months)')

In [ ]:
# auto_arima selects (p,d,q)(P,D,Q) automatically via AIC
# m=3 models quarterly seasonality (Tesla peaks at Q-end)
auto_model = pm.auto_arima(
    ts_train,
    seasonal=True,
    m=3,
    stepwise=True,
    information_criterion='aic',
    error_action='ignore',
    suppress_warnings=True
)
print(auto_model.summary())

In [ ]:
# Forecast: 6 holdout months + 6 future months
N_FUTURE        = HOLDOUT + 6
forecast_vals, conf_int = auto_model.predict(n_periods=N_FUTURE, return_conf_int=True)
forecast_index  = pd.date_range(start=ts_test.index[0], periods=N_FUTURE, freq='MS')

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(ts_train.index, ts_train,  label='Training data',    color='steelblue')
ax.plot(ts_test.index,  ts_test,   label='Actual (holdout)', color='green',  linewidth=2)
ax.plot(forecast_index, forecast_vals, label='Forecast',     color='tomato', linestyle='--')
ax.fill_between(
    forecast_index,
    conf_int[:, 0], conf_int[:, 1],
    alpha=0.2, color='tomato', label='95% confidence interval'
)
ax.axvline(x=ts_test.index[0], color='gray', linestyle=':', linewidth=1.5, label='Forecast start')
ax.set_title('Tesla Delivery Forecast (SARIMA + Confidence Interval)')
ax.set_xlabel('Date')
ax.set_ylabel('Estimated Deliveries')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate ARIMA on holdout
holdout_forecast = forecast_vals[:HOLDOUT]
arima_mae  = mean_absolute_error(ts_test, holdout_forecast)
arima_rmse = np.sqrt(mean_squared_error(ts_test, holdout_forecast))
print(f'ARIMA Holdout MAE  : {arima_mae:,.0f}')
print(f'ARIMA Holdout RMSE : {arima_rmse:,.0f}')

## 12. Save Best Model

In [ ]:
os.makedirs('models', exist_ok=True)   # creates folder if it doesn't exist
joblib.dump(best_pipeline, 'models/tesla_model.pkl')
print('Model saved to models/tesla_model.pkl')

## 13. Summary

| Stage | Key decision |
|---|---|
| Preprocessing | Median imputation; train/test split before any encoding |
| Feature engineering | Delivery ratio, price/km, cyclical month, lag-1, rolling mean |
| Encoding | `OneHotEncoder` inside `ColumnTransformer` — no leakage |
| Scaling | `StandardScaler` applied only to numerical columns, train-fit only |
| Models | Linear Regression vs Random Forest vs Tuned RF |
| Tuning | `GridSearchCV` with `cv=5`, scored on RMSE |
| Time series | SARIMA via `auto_arima`, holdout split, confidence intervals |
